In [55]:
import os, glob, subprocess
from collections import defaultdict
import pandas as pd

# Make mismatch ratio matrix for NanoNASC-seq

In [93]:
for ct in ["K562", "mESC"]:
    for mt in ["raw", "corrected", "linkage"]:
        print("-" * 80)
        print("Cell type:", ct)
        print("Mismatch event:", mt)

        outfile = "results/mismatch_ratio_matrix/NanoNASCseq.%s.s4U_0uM_3h.%s.csv" % (ct, mt)
        if os.path.exists(outfile):
            print("%s exists!" % outfile)
            # continue
            
        info = pd.read_csv("../../1_NanoNASCseq/data/NanoNASCseq.csv")
        info = info[(info["CellLine"] == ct) & (info["s4U"] == 0) & (info["Time"] == 3) & (info["ActD"].isna())]
        print("Cell number:", len(info))
        
        array = []
        for cell in info["Cell"]:
            if mt == "raw":
                path = "../../1_NanoNASCseq/results/4_mismatch/3_ratio_rmdup/%s/%s.tsv" % (cell.split(".")[0], cell)
            elif mt == "corrected":
                path = "../../1_NanoNASCseq/results/4_mismatch/4_ratio_consensus/%s/%s.tsv" % (cell.split(".")[0], cell)
            elif mt == "linkage":
                path = "../../1_NanoNASCseq/results/4_mismatch/5_ratio_consensus_linkage/%s/%s.tsv" % (cell.split(".")[0], cell)
            else:
                assert False
            d = pd.read_csv(path, sep="\t", index_col=0)
            d = d[["-" not in t for t in d.index]]
                
            if d["BaseCount"]["TC"] < 1e6:
                print("Warning: %s has low T number, removed!" % cell)
                continue
            s = d["Ratio"]
            s.name = cell
            array.append(s)
        m = pd.DataFrame(array)
        m.columns = ["%s_ratio" % x for x  in m.columns]
        m.index.name = "Cell"
        m.to_csv(outfile)
        print("Final cell number:", len(m))

--------------------------------------------------------------------------------
Cell type: K562
Mismatch event: raw
results/mismatch_ratio_matrix/NanoNASCseq.K562.s4U_0uM_3h.raw.csv exists!
Cell number: 196
Final cell number: 190
--------------------------------------------------------------------------------
Cell type: K562
Mismatch event: corrected
results/mismatch_ratio_matrix/NanoNASCseq.K562.s4U_0uM_3h.corrected.csv exists!
Cell number: 196
Final cell number: 182
--------------------------------------------------------------------------------
Cell type: K562
Mismatch event: linkage
results/mismatch_ratio_matrix/NanoNASCseq.K562.s4U_0uM_3h.linkage.csv exists!
Cell number: 196
Final cell number: 182
--------------------------------------------------------------------------------
Cell type: mESC
Mismatch event: raw
results/mismatch_ratio_matrix/NanoNASCseq.mESC.s4U_0uM_3h.raw.csv exists!
Cell number: 64
Final cell number: 64
----------------------------------------------------------

# Make mismatch ratio matrix for NASC-seq

In [97]:
for group in ["Tanglab", "GSE128273"]:
    print("-" * 80)
    print("Source:", group)

    if group == "Tanglab":
        pattern = "../../2_NASCseq/results/3_mismatch/3_mismatch_ratio/*/20*.K562_s4U_0uM_180min.tsv"
        outfile = "results/mismatch_ratio_matrix/NASCseq.K562.s4U_0uM_3h.csv"
    elif group == "GSE128273":
        pattern = "../../2_NASCseq/results/3_mismatch/3_mismatch_ratio/*/GSE128273_NASCseq*.K562_s4U_0uM_180min.tsv"
        outfile = "results/mismatch_ratio_matrix/GSE128273_NASCseq.K562.s4U_0uM_3h.csv"
    else:
        assert False

    if os.path.exists(outfile):
        print("%s exists!" % outfile)
        # continue
    
    array = []
    for path in sorted(glob.glob(pattern)):
        cell = os.path.splitext(os.path.basename(path))[0]
        d = pd.read_csv(path, sep="\t", index_col=0)
        d = d[["-" not in t for t in d.index]]
        if d["BaseCount"]["TC"] < 1e6:
            print("Warning: %s has low T number, removed!" % cell)
            continue
        s = d["Ratio"]
        s.name = cell
        array.append(s)
    m = pd.DataFrame(array)
    m.columns = ["%s_ratio" % x for x  in m.columns]
    m.index.name = "Cell"
    m.to_csv(outfile)
    print("Final cell number:", len(m))

--------------------------------------------------------------------------------
Source: Tanglab
results/mismatch_ratio_matrix/NASCseq.K562.s4U_0uM_3h.csv exists!
Final cell number: 39
--------------------------------------------------------------------------------
Source: GSE128273
results/mismatch_ratio_matrix/GSE128273_NASCseq.K562.s4U_0uM_3h.csv exists!
Final cell number: 16


# Train Pe Model

In [98]:
for path in sorted(glob.glob("results/mismatch_ratio_matrix/*.csv")):
    path2 = "results/models/%s.pkl" % path.split("/")[-1][:-4]
    log = path2[:-4] + ".log"
    print("-" * 80)
    print("Matrix file:", path)
    print("Outfile:", path2)
    
    if os.path.exists(path2):
        print("%s exists!" % path2)
        
    cmd = "nasctools TrainPeModel -i %s -o %s > %s" % (path, path2, log)
    print(cmd)
    subprocess.check_call(cmd, shell=True)

--------------------------------------------------------------------------------
Matrix file: results/mismatch_ratio_matrix/GSE128273_NASCseq.K562.s4U_0uM_3h.csv
Outfile: results/models/GSE128273_NASCseq.K562.s4U_0uM_3h.pkl
results/models/GSE128273_NASCseq.K562.s4U_0uM_3h.pkl exists!
nasctools TrainPeModel -i results/mismatch_ratio_matrix/GSE128273_NASCseq.K562.s4U_0uM_3h.csv -o results/models/GSE128273_NASCseq.K562.s4U_0uM_3h.pkl > results/models/GSE128273_NASCseq.K562.s4U_0uM_3h.log
--------------------------------------------------------------------------------
Matrix file: results/mismatch_ratio_matrix/NASCseq.K562.s4U_0uM_3h.csv
Outfile: results/models/NASCseq.K562.s4U_0uM_3h.pkl
results/models/NASCseq.K562.s4U_0uM_3h.pkl exists!
nasctools TrainPeModel -i results/mismatch_ratio_matrix/NASCseq.K562.s4U_0uM_3h.csv -o results/models/NASCseq.K562.s4U_0uM_3h.pkl > results/models/NASCseq.K562.s4U_0uM_3h.log
--------------------------------------------------------------------------------